# Задание 1: Реализация PWM с нуля

Реализую построение PFM → PPM → PWM, не используя биоинформатические библиотеки.

In [7]:
import numpy as np
import math

In [8]:
sites = [
    "GAGGTAAAC",
    "TCCGTAAGC",
    "CAGGTTGGA",
    "ACAGTCAGC",
    "TAGGTCAGC",
    "CAGGTCAGC",
    "CAGGTCGAT",
    "CAGGTCAGC",
    "CAGGTCAGC",
    "CAGGTTGGC"
]

nucleotides = ['A', 'C', 'G', 'T']
N = len(sites)     # число последовательностей = 10
L = len(sites[0])  # длина мотива = 9
alpha = 0.1        # псевдосчёт

print(f"N = {N}, L = {L}")

N = 10, L = 9


## 1. PFM — Position Frequency Matrix

$$c_{a,i} = |\{j : s_j[i] = a\}|$$

Строки: A, C, G, T. Столбцы: позиции 0..L-1.

In [9]:
def build_pfm(sites):
    """
    Numpy-функция. Возвращает PFM размером (4, L).
    Строки: A, C, G, T.
    """
    L = len(sites[0])
    pfm = np.zeros((4, L), dtype=int)
    nuc_index = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

    for seq in sites:
        for pos in range(L):
            pfm[nuc_index[seq[pos]], pos] += 1

    return pfm


pfm = build_pfm(sites)

print("Итоговая матрица частот (PFM):")
print("Строки: A, C, G, T")
print(pfm)

Итоговая матрица частот (PFM):
Строки: A, C, G, T
[[ 1  8  1  0  0  2  7  2  1]
 [ 6  2  1  0  0  6  0  0  8]
 [ 1  0  8 10  0  0  3  8  0]
 [ 2  0  0  0 10  2  0  0  1]]


## 2. PPM — Position Probability Matrix

$$p_{a,i} = \frac{c_{a,i} + \alpha}{\sum_b (c_{b,i} + \alpha)}$$

Псевдосчёт $\alpha = 0.1$ защищает от нулевых вероятностей.

In [10]:
# PPM считаем поколоночно — явно, как на семинаре
ppm = [[0] * L for _ in range(4)]

for pos in range(L):
    count_A = pfm[0][pos] + alpha
    count_C = pfm[1][pos] + alpha
    count_G = pfm[2][pos] + alpha
    count_T = pfm[3][pos] + alpha

    total = count_A + count_C + count_G + count_T

    ppm[0][pos] = count_A / total
    ppm[1][pos] = count_C / total
    ppm[2][pos] = count_G / total
    ppm[3][pos] = count_T / total

print("PPM (Position Probability Matrix):")
print("Строки: A, C, G, T")
for i, nuc in enumerate(nucleotides):
    print(f"{nuc}: {[round(x, 4) for x in ppm[i]]}")

PPM (Position Probability Matrix):
Строки: A, C, G, T
A: [np.float64(0.1058), np.float64(0.7788), np.float64(0.1058), np.float64(0.0096), np.float64(0.0096), np.float64(0.2019), np.float64(0.6827), np.float64(0.2019), np.float64(0.1058)]
C: [np.float64(0.5865), np.float64(0.2019), np.float64(0.1058), np.float64(0.0096), np.float64(0.0096), np.float64(0.5865), np.float64(0.0096), np.float64(0.0096), np.float64(0.7788)]
G: [np.float64(0.1058), np.float64(0.0096), np.float64(0.7788), np.float64(0.9712), np.float64(0.0096), np.float64(0.0096), np.float64(0.2981), np.float64(0.7788), np.float64(0.0096)]
T: [np.float64(0.2019), np.float64(0.0096), np.float64(0.0096), np.float64(0.0096), np.float64(0.9712), np.float64(0.2019), np.float64(0.0096), np.float64(0.0096), np.float64(0.1058)]


## 3. PWM — Position Weight Matrix

$$w_{a,i} = \log_2 \frac{p_{a,i}}{b_a}$$

Фоновые частоты: $P(A) = P(T) = 0.295$, $P(G) = P(C) = 0.205$.

In [11]:
bg_A = 0.295
bg_C = 0.205
bg_G = 0.205
bg_T = 0.295

pwm = []
for row in range(4):
    new_row = []
    for col in range(L):
        new_row.append(0)
    pwm.append(new_row)

for pos in range(L):
    pwm[0][pos] = math.log2(ppm[0][pos] / bg_A)
    pwm[1][pos] = math.log2(ppm[1][pos] / bg_C)
    pwm[2][pos] = math.log2(ppm[2][pos] / bg_G)
    pwm[3][pos] = math.log2(ppm[3][pos] / bg_T)

print("PWM (Position Weight Matrix):")
print(f"A: {[round(x, 3) for x in pwm[0]]}")
print(f"C: {[round(x, 3) for x in pwm[1]]}")
print(f"G: {[round(x, 3) for x in pwm[2]]}")
print(f"T: {[round(x, 3) for x in pwm[3]]}")

PWM (Position Weight Matrix):
A: [-1.48, 1.401, -1.48, -4.939, -4.939, -0.547, 1.211, -0.547, -1.48]
C: [1.517, -0.022, -0.955, -4.414, -4.414, 1.517, -4.414, -4.414, 1.926]
G: [-0.955, -4.414, 1.926, 2.244, -4.414, -4.414, 0.54, 1.926, -4.414]
T: [-0.547, -4.939, -4.939, -4.939, 1.719, -0.547, -4.939, -4.939, -1.48]


## 4. Максимальный и минимальный скор

$$S(x) = \sum_{i=1}^{L} w_{x_i,\, i}$$

In [12]:
max_score = 0
min_score = 0
best_seq = ""
worst_seq = ""

for pos in range(L):
    w_A = pwm[0][pos]
    w_C = pwm[1][pos]
    w_G = pwm[2][pos]
    w_T = pwm[3][pos]

    # Нуклеотид с максимальным весом
    best_weight = w_A
    best_nuc = 'A'
    if w_C > best_weight:
        best_weight = w_C
        best_nuc = 'C'
    if w_G > best_weight:
        best_weight = w_G
        best_nuc = 'G'
    if w_T > best_weight:
        best_weight = w_T
        best_nuc = 'T'
    best_seq += best_nuc
    max_score += best_weight

    # Нуклеотид с минимальным весом
    worst_weight = w_A
    worst_nuc = 'A'
    if w_C < worst_weight:
        worst_weight = w_C
        worst_nuc = 'C'
    if w_G < worst_weight:
        worst_weight = w_G
        worst_nuc = 'G'
    if w_T < worst_weight:
        worst_weight = w_T
        worst_nuc = 'T'
    worst_seq += worst_nuc
    min_score += worst_weight

print(f"Макс скор: {max_score:.3f}, последовательность: {best_seq}")
print(f"Мин скор:  {min_score:.3f}, последовательность: {worst_seq}")

Макс скор: 15.385, последовательность: CAGGTCAGC
Мин скор:  -39.943, последовательность: ATTAAGTTG
